# Win Prediction — Composition + Map

This notebook extends the comp-only model from notebook 03 by adding map as a feature, testing whether knowing the specific map improves the model's ability to predict win probability from team composition.

In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import joblib

In [2]:
df_raw = pd.read_csv('../data/processed/comp_map.csv')
df = df_raw.copy()

In [3]:
df.head()

,match_id,map_name,Controller,Duelist,Initiator,Sentinel,won
0,427991,Pearl,1,2,1,1,0
1,427991,Split,2,1,1,1,0
2,427991,Pearl,1,1,2,1,1
3,427991,Split,1,1,2,1,1
4,427992,Bind,2,1,2,0,1


In [4]:
df.shape

(2540, 7)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2540 entries, 0 to 2539
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   match_id    2540 non-null   int64
 1   map_name    2540 non-null   str  
 2   Controller  2540 non-null   int64
 3   Duelist     2540 non-null   int64
 4   Initiator   2540 non-null   int64
 5   Sentinel    2540 non-null   int64
 6   won         2540 non-null   int64
dtypes: int64(6), str(1)
memory usage: 152.7 KB


In [6]:
df.isna().sum()

match_id      0
map_name      0
Controller    0
Duelist       0
Initiator     0
Sentinel      0
won           0
dtype: int64

`map_name` is categorical text, which a model can't use directly. One-hot encoding converts it into separate 0/1 columns, one per map. `drop_first=True` drops one map as an implicit baseline, avoiding redundant, perfectly correlated columns that can destabilize a linear model's coefficients (the "dummy variable trap").

In [7]:
df_encoded = pd.get_dummies(df, columns=["map_name"], drop_first=True)

In [8]:
df_encoded.columns.tolist()

['match_id',
 'Controller',
 'Duelist',
 'Initiator',
 'Sentinel',
 'won',
 'map_name_Ascent',
 'map_name_Bind',
 'map_name_Corrode',
 'map_name_Fracture',
 'map_name_Haven',
 'map_name_Icebox',
 'map_name_Lotus',
 'map_name_Pearl',
 'map_name_Split',
 'map_name_Sunset']

### Train/test split methodology

Same rationale as notebook 03: each match produces two mirrored team rows, so a naive random split risks leaking match information across train/test. The split below groups on `match_id` (not `match_id` + `map_name`) using `GroupShuffleSplit`, since two maps within the same series aren't independent either — they share the same two teams and relative skill gap. Grouping at the match/series level is the more conservative choice.

In [11]:
X = df_encoded.drop(columns=['match_id', 'won'])
y = df_encoded['won']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=20)
train_idx, test_idx = next(gss.split(X, y, groups=df['match_id']))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [12]:
model = LogisticRegression()
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

Note: because of the one-hot encoding, each map's coefficient represents its effect *relative to the dropped baseline map (Abyss)*, not an absolute standalone effect. A positive map coefficient means "better than baseline, holding composition constant" — not "this map is good in isolation."

In [13]:
for feature, weight in zip(X.columns, model.coef_[0]):
    print(f"{feature} Coef Weight: {weight:.4f}")

print(f"\nThe model intercept is: {model.intercept_[0]:.7f}")

Controller Coef Weight: 0.0523
Duelist Coef Weight: -0.0001
Initiator Coef Weight: -0.0931
Sentinel Coef Weight: 0.0736
map_name_Ascent Coef Weight: 0.0014
map_name_Bind Coef Weight: 0.0128
map_name_Corrode Coef Weight: -0.0387
map_name_Fracture Coef Weight: 0.0473
map_name_Haven Coef Weight: -0.0175
map_name_Icebox Coef Weight: -0.0261
map_name_Lotus Coef Weight: -0.0181
map_name_Pearl Coef Weight: -0.0098
map_name_Split Coef Weight: 0.0056
map_name_Sunset Coef Weight: -0.0197

The model intercept is: 0.0069553


In [14]:
y_pred = model.predict(X_test)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5195


In [16]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives: {tn} ({tn/len(y_test)*100:.1f}%)")
print(f"False Positives: {fp} ({fp/len(y_test)*100:.1f}%)")
print(f"False Negatives: {fn} ({fn/len(y_test)*100:.1f}%)")
print(f"True Positives: {tp} ({tp/len(y_test)*100:.1f}%)")

True Negatives: 125 (24.3%)
False Positives: 132 (25.7%)
False Negatives: 115 (22.4%)
True Positives: 142 (27.6%)


**Takeaway:** Adding map as a feature achieved 51.95% accuracy on a match-grouped train/test split (`GroupShuffleSplit` on `match_id`) — identical to notebook 03's comp-only accuracy of 51.95%. This is a meaningful result in itself: map context adds no measurable predictive power beyond composition alone. The role coefficients remained small and consistent with notebook 03 (ranging from -0.09 to +0.07), and every map coefficient was similarly small in magnitude (from -0.04 for Corrode to +0.05 for Fracture, relative to the Abyss baseline) — no single map showed a decisive effect on win probability, holding composition constant. The confusion matrix shows no directional bias, with errors and correct predictions spread fairly evenly across all four outcomes (22-28% each).

Taken together with notebook 03, this reinforces a consistent conclusion: neither team role composition nor map context, alone or combined, meaningfully predicts match outcomes — reinforcing that player skill, coordination, and execution are almost certainly the dominant factors, none of which either model accounts for.

In [17]:
joblib.dump(model, '../models/comp_map_model.joblib')

['../models/comp_map_model.joblib']